In [1]:
import networkx as nx
import numpy as np
from gwgraphs import *

In [2]:
def read_graph_data():
    # Read the adjacency matrix (edges)
    with open("BZR_A.txt", "r") as f:
        edges = [tuple(map(int, line.split(','))) for line in f]

    # Read graph indicators
    with open("BZR_graph_indicator.txt", "r") as f:
        graph_indicators = [int(line.strip()) for line in f if line.strip()]

    # Read graph labels
    with open("BZR_graph_labels.txt", "r") as f:
        graph_labels = [int(line.strip()) for line in f]

    # Read node labels
    with open("BZR_node_labels.txt", "r") as f:
        node_labels = [int(line.strip()) for line in f]

    # Read node attributes
    with open("BZR_node_attributes.txt", "r") as f:
        node_attributes = [
            list(map(float, line.strip().split(','))) for line in f if line.strip()
        ]

    return edges, graph_indicators, graph_labels, node_labels, node_attributes


def construct_networkx_graph(edges, graph_indicators, graph_labels,
                             node_labels, node_attributes):
    graphs = {}
    graphs_normalized = {}
    
    # Normalize all node attributes
    normalized_attributes = normalize_features(node_attributes)
    
    for node_id, graph_id in enumerate(graph_indicators, start=1):
        if graph_id not in graphs:
            graphs[graph_id] = nx.Graph(label=graph_labels[graph_id - 1])
            graphs_normalized[graph_id] = nx.Graph(label=graph_labels[graph_id - 1])

    for edge in edges:
        node1, node2 = edge
        graph_id = graph_indicators[
            node1 - 1]  # Determine which graph the edge belongs to
        graphs[graph_id].add_edge(node1, node2)
        graphs_normalized[graph_id].add_edge(node1, node2)

    for node_id, (graph_id, node_label,
                  node_attr, norm_attr) in enumerate(zip(graph_indicators, node_labels,
                                              node_attributes, normalized_attributes),
                                          start=1):
        # Original graphs with original features
        graphs[graph_id].nodes[node_id]['label'] = node_label
        graphs[graph_id].nodes[node_id]['attributes'] = node_attr
        
        # Normalized graphs with normalized features
        graphs_normalized[graph_id].nodes[node_id]['label'] = node_label
        graphs_normalized[graph_id].nodes[node_id]['attributes'] = norm_attr

    return graphs, graphs_normalized


def compute_bzr_distance_matrix(graph):
    num_nodes = len(graph.nodes)
    distance_matrix = np.full((num_nodes, num_nodes), 10000.0)

    # Get the correct node indices to be used for indexing the matrix
    node_list = list(graph.nodes())
    node_index = {node: i for i, node in enumerate(node_list)}

    for node in graph.nodes:
        lengths = nx.shortest_path_length(graph, source=node, weight="weight")

        for target, length in lengths.items():
            if target in node_index:
                distance_matrix[node_index[node], node_index[target]] = length

    return distance_matrix


def normalize_features(features):
    features = np.array(features)
    # Normalize each feature dimension independently
    mean = np.mean(features, axis=0)
    std = np.std(features, axis=0)
    # Avoid division by zero
    std[std == 0] = 1
    normalized_features = (features - mean) / std
    return normalized_features.tolist()

In [3]:
edges, graph_indicators, graph_labels, node_labels, node_attributes = read_graph_data()
graphs, graphs_normalized = construct_networkx_graph(edges, graph_indicators, graph_labels,
                                                   node_labels, node_attributes)
for graph_id, graph in graphs.items():
    print(
        f"Graph ID: {graph_id}, number of nodes: {graph.number_of_nodes()}, label: {graph.graph['label']}"
        )

Graph ID: 1, number of nodes: 30, label: -1
Graph ID: 2, number of nodes: 33, label: -1
Graph ID: 3, number of nodes: 30, label: -1
Graph ID: 4, number of nodes: 33, label: -1
Graph ID: 5, number of nodes: 30, label: -1
Graph ID: 6, number of nodes: 33, label: -1
Graph ID: 7, number of nodes: 30, label: -1
Graph ID: 8, number of nodes: 30, label: -1
Graph ID: 9, number of nodes: 33, label: -1
Graph ID: 10, number of nodes: 30, label: -1
Graph ID: 11, number of nodes: 30, label: -1
Graph ID: 12, number of nodes: 30, label: -1
Graph ID: 13, number of nodes: 33, label: -1
Graph ID: 14, number of nodes: 33, label: -1
Graph ID: 15, number of nodes: 33, label: -1
Graph ID: 16, number of nodes: 33, label: -1
Graph ID: 17, number of nodes: 32, label: -1
Graph ID: 18, number of nodes: 32, label: -1
Graph ID: 19, number of nodes: 32, label: -1
Graph ID: 20, number of nodes: 35, label: -1
Graph ID: 21, number of nodes: 32, label: -1
Graph ID: 22, number of nodes: 35, label: -1
Graph ID: 23, numbe

In [4]:
graphs_list = []
labels_list = []
for graph_id, graph in graphs.items():
    # Add the graph object to the list
    graphs_list.append(graph)

    # Extract the label and add to the label list
    label = graph.graph['label']
    labels_list.append(label)

# Convert graphs into pairwise distance matrices
distance_matrices = [compute_bzr_distance_matrix(graph) for graph in graphs_list]

In [5]:
# ATTEMPT 1: Use the dataset's average # of nodes for the barycenters
bzr_exp = fgw_barycenter_class_alg_var_nodes(distance_matrices, 
                                             graphs_list, 
                                             labels_list, 
                                             num_folds=10, 
                                             num_nodes_negone=36, 
                                             num_nodes_one=36,
                                             method="stratk")

Stratified k-fold CV selected
Fold 0:
GW barycenter for Class -1 graphs computed
GW barycenter for Class 1 graphs computed
Distance between barycenters:  0.6782577054662031  Fold 0
F1 Weighted: 77.0140428677014%, Fold 0
Classification Accuracy: 75.60975609756098% Fold 0
Classification Accuracy for Class -1: 78.125% Fold 0
Classification Accuracy for Class 1: 66.66666666666666% Fold 0
Fold 1:
GW barycenter for Class -1 graphs computed
GW barycenter for Class 1 graphs computed
Distance between barycenters:  0.8349221478024608  Fold 1
F1 Weighted: 66.55664344541948%, Fold 1
Classification Accuracy: 63.41463414634146% Fold 1
Classification Accuracy for Class -1: 59.375% Fold 1
Classification Accuracy for Class 1: 77.77777777777779% Fold 1
Fold 2:
GW barycenter for Class -1 graphs computed
GW barycenter for Class 1 graphs computed
Distance between barycenters:  0.843079876918166  Fold 2
F1 Weighted: 70.39560005032622%, Fold 2
Classification Accuracy: 68.29268292682927% Fold 2
Classification

In [6]:
# ATTEMPT 2: Use the average # of nodes for each class to make the barycenters for each class
bzr_exp2 = bzr_exp = fgw_barycenter_class_alg_var_nodes(distance_matrices, 
                                             graphs_list, 
                                             labels_list, 
                                             num_folds=10, 
                                             num_nodes_negone=37, 
                                             num_nodes_one=33,
                                             method="stratk")

Stratified k-fold CV selected
Fold 0:
GW barycenter for Class -1 graphs computed
GW barycenter for Class 1 graphs computed
Distance between barycenters:  0.9440610512629073  Fold 0
F1 Weighted: 77.0140428677014%, Fold 0
Classification Accuracy: 75.60975609756098% Fold 0
Classification Accuracy for Class -1: 78.125% Fold 0
Classification Accuracy for Class 1: 66.66666666666666% Fold 0
Fold 1:
GW barycenter for Class -1 graphs computed
GW barycenter for Class 1 graphs computed
Distance between barycenters:  0.9542426727197347  Fold 1
F1 Weighted: 64.33088140405215%, Fold 1
Classification Accuracy: 60.97560975609756% Fold 1
Classification Accuracy for Class -1: 59.375% Fold 1
Classification Accuracy for Class 1: 66.66666666666666% Fold 1
Fold 2:
GW barycenter for Class -1 graphs computed
GW barycenter for Class 1 graphs computed
Distance between barycenters:  0.9724466025981471  Fold 2
F1 Weighted: 70.39560005032622%, Fold 2
Classification Accuracy: 68.29268292682927% Fold 2
Classificatio

In [7]:
### Normalized feature vector experiment ###
graphs_list_norm = []
labels_list_norm = []
for graph_id, graph in graphs_normalized.items():
    # Add the graph object to the list
    graphs_list_norm.append(graph)

    # Extract the label and add to the label list
    label = graph.graph['label']
    labels_list_norm.append(label)

# Convert graphs into pairwise distance matrices
distance_matrices_norm = [compute_bzr_distance_matrix(graph) for graph in graphs_list_norm]

In [8]:
bzr_exp3 = fgw_barycenter_class_alg_var_nodes(distance_matrices_norm, 
                                             graphs_list_norm, 
                                             labels_list_norm, 
                                             num_folds=10, 
                                             num_nodes_negone=36, 
                                             num_nodes_one=36,
                                             method="stratk")

Stratified k-fold CV selected
Fold 0:
GW barycenter for Class -1 graphs computed
GW barycenter for Class 1 graphs computed
Distance between barycenters:  0.5646899413923518  Fold 0
F1 Weighted: 70.78476679503638%, Fold 0
Classification Accuracy: 68.29268292682927% Fold 0
Classification Accuracy for Class -1: 68.75% Fold 0
Classification Accuracy for Class 1: 66.66666666666666% Fold 0
Fold 1:
GW barycenter for Class -1 graphs computed
GW barycenter for Class 1 graphs computed
Distance between barycenters:  0.5102338947507263  Fold 1
F1 Weighted: 66.51884700665188%, Fold 1
Classification Accuracy: 63.41463414634146% Fold 1
Classification Accuracy for Class -1: 62.5% Fold 1
Classification Accuracy for Class 1: 66.66666666666666% Fold 1
Fold 2:
GW barycenter for Class -1 graphs computed
GW barycenter for Class 1 graphs computed
Distance between barycenters:  0.5404821200526391  Fold 2
F1 Weighted: 69.7778031644485%, Fold 2
Classification Accuracy: 68.29268292682927% Fold 2
Classification A

In [10]:
bzr_exp4 = fgw_barycenter_class_alg_var_nodes(distance_matrices_norm, 
                                             graphs_list_norm, 
                                             labels_list_norm, 
                                             num_folds=10, 
                                             num_nodes_negone=37, 
                                             num_nodes_one=33,
                                             method="stratk")

Stratified k-fold CV selected
Fold 0:
GW barycenter for Class -1 graphs computed
GW barycenter for Class 1 graphs computed
Distance between barycenters:  0.8245450212794687  Fold 0
F1 Weighted: 70.78476679503638%, Fold 0
Classification Accuracy: 68.29268292682927% Fold 0
Classification Accuracy for Class -1: 68.75% Fold 0
Classification Accuracy for Class 1: 66.66666666666666% Fold 0
Fold 1:
GW barycenter for Class -1 graphs computed
GW barycenter for Class 1 graphs computed
Distance between barycenters:  0.7839605569414733  Fold 1
F1 Weighted: 68.78952122854561%, Fold 1
Classification Accuracy: 65.85365853658537% Fold 1
Classification Accuracy for Class -1: 62.5% Fold 1
Classification Accuracy for Class 1: 77.77777777777779% Fold 1
Fold 2:
GW barycenter for Class -1 graphs computed
GW barycenter for Class 1 graphs computed
Distance between barycenters:  0.877495915173056  Fold 2
F1 Weighted: 69.7778031644485%, Fold 2
Classification Accuracy: 68.29268292682927% Fold 2
Classification Ac